In [1]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约1-2分钟）
!pip install -q numpy scipy matplotlib soundfile torch torchaudio
!apt-get install -qq ffmpeg sox
print('✅ 环境配置完成！')

Selecting previously unselected package libopencore-amrnb0:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../0-libopencore-amrnb0_0.1.5-1_amd64.deb ...
Unpacking libopencore-amrnb0:amd64 (0.1.5-1) ...
Selecting previously unselected package libopencore-amrwb0:amd64.
Preparing to unpack .../1-libopencore-amrwb0_0.1.5-1_amd64.deb ...
Unpacking libopencore-amrwb0:amd64 (0.1.5-1) ...
Selecting previously unselected package libsox3:amd64.
Preparing to unpack .../2-libsox3_14.4.2+git20190427-2+deb11u2ubuntu0.22.04.1_amd64.deb ...
Unpacking libsox3:amd64 (14.4.2+git20190427-2+deb11u2ubuntu0.22.04.1) ...
Selecting previously unselected package libsox-fmt-alsa:amd64.
Preparing to unpack .../3-libsox-fmt-alsa_14.4.2+git20190427-2+deb11u2ubuntu0.22.04.1_amd64.deb ...
Unpacking libsox-fmt-alsa:amd64 (14.4.2+git20190427-2+deb11u2ubuntu0.22.04.1) ...
Selecting previously unselected package libwavpack1:amd64.
Preparing to unpack .../4-libwavpack1_

# Python进阶——面向对象与代码组织

## 学习目标

- 理解从"函数+字典"到"类"的过渡
- 能读懂别人写的类，能写简单的类
- 理解继承的概念
- 理解模块/包的概念，知道import做了什么
- **关键衔接**：读懂PyTorch源码中的类（nn.Module, Dataset）


## 1. 从"函数+字典"到"类"

假设我们要管理一段音频信号，需要：
- 存储波形数据、采样率等信息
- 提供播放、画图、计算频谱等方法

先来看两种不同的实现方式。

### 1.1 方式一：函数 + 字典

数据（字典）和操作（函数）是分开的。

In [2]:
# ===== 方式一：函数 + 字典 =====
import numpy as np
def create_signal(waveform, sample_rate, label=''):
    """创建一个信号字典"""
    return {
        'waveform': waveform,
        'sample_rate': sample_rate,
        'label': label,
    }

def plot_signal(signal_dict):
    """画出信号"""
    sr = signal_dict['sample_rate']
    t = np.linspace(0, len(signal_dict['waveform'])/sr, len(signal_dict['waveform']), endpoint=False)
    plt.figure(figsize=(12, 4))
    plt.plot(t, signal_dict['waveform'])
    plt.title(signal_dict['label'])
    plt.xlabel('Time (s)')
    plt.show()

def compute_spectrum(signal_dict):
    """计算频谱"""
    N = len(signal_dict['waveform'])
    spectrum = np.abs(np.fft.rfft(signal_dict['waveform']))
    freqs = np.fft.rfftfreq(N, 1/signal_dict['sample_rate'])
    return freqs, spectrum

# 使用
t = np.linspace(0, 1, 16000, endpoint=False)
wave = 0.5 * np.sin(2 * np.pi * 440 * t)
sig = create_signal(wave, 16000, '440 Hz Sine Wave')
plot_signal(sig)

NameError: name 'np' is not defined

### 1.2 方式二：类

现在用类来做同样的事。**数据和方法绑在一起**。

In [ ]:
# ===== 方式二：类 =====

class Signal:
    """音频信号类"""

    def __init__(self, waveform, sample_rate, label=''):
        """初始化方法（构造函数）

        self.waveform = waveform    # 波形数据
        self.sample_rate = sample_rate  # 采样率
        self.label = label          # 标签

    def plot(self):
        """画出信号"""
        sr = self.sample_rate
        t = np.linspace(0, len(self.waveform)/sr, len(self.waveform), endpoint=False)
        plt.figure(figsize=(12, 4))
        plt.plot(t, self.waveform)
        plt.title(self.label)
        plt.show()

    def compute_spectrum(self):
        """计算频谱"""
        N = len(self.waveform)
        spectrum = np.abs(np.fft.rfft(self.waveform))
        freqs = np.fft.rfftfreq(N, 1/self.sample_rate)
        return freqs, spectrum

# 使用
sig = Signal(wave, 16000, '440 Hz Sine Wave')
sig.plot()

### 对比

| 方式 | 调用方式 | 问题 |
|------|---------|------|
| 函数+字典 | `plot_signal(sig)` | 函数和数据是分开的，需要把字典传给函数 |
| 类 | `sig.plot()` | 数据和方法绑在一起，调用更自然 |

**类的核心思想**：数据（属性）和操作数据的方法绑在一起。

- `self.waveform` 是属性（存储数据）
- `self.plot()` 是方法（操作数据）
- `self` 就是实例本身


## 2. 继承——在已有的东西上加修改

继承是面向对象中最重要的概念之一。它的直觉很简单：

> **继承就是在已有的东西上加修改。**

比如：
- `Signal` 是基类，能存储波形、画图
- `SineWave` 继承 `Signal`，但专门生成正弦波
- `NoisySignal` 继承 `Signal`，但在初始化时额外加了噪声


In [ ]:
class SineWave(Signal):
    """正弦波信号——继承Signal，专门生成正弦波"""

    def __init__(self, frequency, amplitude=0.5, duration=1.0, sample_rate=16000):
        # 先生成波形
        t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
        waveform = amplitude * np.sin(2 * np.pi * frequency * t)

        # 然后调用父类的__init__
        super().__init__(waveform, sample_rate, f'{frequency} Hz Sine Wave')

        # 子类可以添加自己的属性
        self.frequency = frequency
        self.amplitude = amplitude
        self.duration = duration

# 使用
sine = SineWave(440)
print(f'频率: {sine.frequency} Hz')
print(f'采样率: {sine.sample_rate} Hz')
sine.plot()

In [ ]:
class NoisySignal(Signal):
    """带噪信号——继承Signal，在干净信号上加噪声"""

    def __init__(self, clean_waveform, sample_rate, noise_level=0.1, label=''):
        # 在干净信号上加高斯噪声
        noise = np.random.randn(len(clean_waveform)) * noise_level
        noisy = clean_waveform + noise

        # 调用父类构造函数
        label = label or f'Noisy Signal (noise_level={noise_level})'
        super().__init__(noisy, sample_rate, label)

        # 子类自己的属性
        self.clean_waveform = clean_waveform
        self.noise_level = noise_level

    def compute_snr(self):
        """计算信噪比 (dB)"""
        signal_power = np.mean(self.clean_waveform ** 2)
        noise = self.waveform - self.clean_waveform
        noise_power = np.mean(noise ** 2)
        if noise_power == 0:
            return float('inf')
        return 10 * np.log10(signal_power / noise_power)

# 使用
clean = SineWave(440)
noisy = NoisySignal(clean.waveform, clean.sample_rate, noise_level=0.3)
print(f'SNR: {noisy.compute_snr():.1f} dB')
noisy.plot()

## 3. 为什么PyTorch里全是类？

现在你有了理解PyTorch代码的基础。PyTorch中的所有组件都是类：

- `nn.Module` 是所有神经网络层的基类
- `nn.Linear` 继承 `nn.Module`，实现了线性变换
- `nn.Conv2d` 继承 `nn.Module`，实现了2D卷积

让我们打开PyTorch源码看看。

In [ ]:
# 查看nn.Module的源码
import inspect
import torch.nn as nn

# 只看__init__和forward方法
source = inspect.getsource(nn.Module)
# 打印前30行
for i, line in enumerate(source.split('\n')[:30], 1):
    print(f'{i:3d}: {line}')

### 关键理解

1. `nn.Module` 的 `__init__` 方法里注册子模块和参数
2. `forward` 方法定义了数据如何流过网络
3. 你不需要自己写 `backward`——PyTorch的自动微分系统帮你做了
4. 所有自定义的网络层都继承 `nn.Module`，只需实现 `__init__` 和 `forward`


In [ ]:
# 一个最简单的nn.Module示例
import torch
import torch.nn as nn

class SimpleLinear(nn.Module):
    """最简单的神经网络层：线性变换 y = Wx + b"""

    def __init__(self, in_features, out_features):
        # 第一步：调用父类的__init__
        super().__init__()

        # 第二步：在__init__中定义层
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        # forward定义数据如何流过网络
        return self.linear(x)

# 使用
layer = SimpleLinear(3, 2)
x = torch.randn(5, 3)  # batch_size=5, in_features=3
y = layer(x)
print(f'输入形状: {x.shape}')
print(f'输出形状: {y.shape}')

### nn.Module vs 普通类：有什么区别？

| 特性 | 普通类 | nn.Module |
|------|--------|----------|
| 参数管理 | 手动管理 | `parameters()` 自动收集 |
| GPU转移 | 手动 `.to('cuda')` | 所有子模块自动转移 |
| 保存加载 | 手动pickle | `state_dict()` 自动处理 |
| 梯度计算 | 手动 | 自动（`autograd`） |

**核心**：`nn.Module` 帮你管理了神经网络中所有繁琐的工程细节，让你只需关注 `__init__`（定义层）和 `forward`（定义数据流）。

## 4. 模块与包：从notebook到.py文件

在研究中，我们经常需要把notebook中的代码整理成可复用的模块。

### 4.1 为什么需要模块？

- notebook适合探索和可视化
- 当代码需要在多个实验中复用时，需要抽取到.py文件
- PyTorch的Dataset、模型定义等，通常放在单独的.py文件中

In [ ]:
# 演示：import一个模块
import os

# 假设我们有一个文件 utils/signal_utils.py
# 其中定义了 Signal 类
#
# utils/
# ├── __init__.py
# └── signal_utils.py
#
# 在其他文件中可以这样使用：
# from utils.signal_utils import Signal
# from utils.signal_utils import SineWave

print('模块导入演示完成')

### 4.2 import做了什么？

```python
from utils.signal_utils import Signal
```

这行代码做了三件事：
1. 找到 `utils/signal_utils.py` 文件
2. 执行文件中的所有代码
3. 把 `Signal` 类引入当前命名空间

这就是为什么 `__init__.py` 很重要——它告诉Python这个目录是一个包。

## 5. 实战：读懂PyTorch Dataset源码

PyTorch的 `Dataset` 类是深度学习数据加载的基础。让我们看看它的源码。

In [ ]:
# 查看Dataset的源码
from torch.utils.data import Dataset
import inspect

source = inspect.getsource(Dataset)
print(source)

源码非常简单——只有3个方法：

1. `__getitem__`：根据索引返回一个样本
2. `__len__`：返回数据集的大小
3. `__init__`：初始化（抽象方法，子类必须实现）

这就是PyTorch数据加载的基石！你只需要实现这三个方法，`DataLoader` 就会自动处理批量加载、数据打乱、多进程加载等所有细节。

## 6. 编程练习：写一个AudioDataset类

**任务**：写一个 `AudioDataset` 类，继承 `Dataset`，实现：
1. `__init__`：接收文件列表和标签列表
2. `__len__`：返回数据集大小
3. `__getitem__`：根据索引加载音频文件并返回（波形, 标签）

**这是模块3中Dataset/DataLoader的前置练习**。

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import soundfile as sf
import os

class AudioDataset(Dataset):
    """音频数据集

    参数:
        file_list: 音频文件路径列表
        label_list: 对应的标签列表
        sample_rate: 目标采样率（如果需要重采样）
    """

    def __init__(self, file_list, label_list, sample_rate=16000):
        # TODO: 验证输入
        assert len(file_list) == len(label_list)

        # TODO: 保存属性
        self.file_list = file_list
        self.label_list = label_list
        self.sample_rate = sample_rate

    def __len__(self):
        # TODO: 返回数据集大小
        return len(self.file_list)

    def __getitem__(self, idx):
        # TODO: 根据索引获取文件路径和标签
        filepath = self.file_list[idx]
        label = self.label_list[idx]

        # TODO: 加载音频文件
        waveform, sr = sf.read(filepath)

        # TODO: 转为torch张量
        waveform_tensor = torch.FloatTensor(waveform)
        label_tensor = torch.LongTensor([label])[0]

        return waveform_tensor, label_tensor

# 演示使用方式（不需要真实文件）
print('AudioDataset类定义完成')
print(f'类继承关系: {AudioDataset.__mro__}')

### DataLoader的使用

有了Dataset，DataLoader会自动处理：
- 批量加载（batch_size）
- 数据打乱（shuffle）
- 多进程加载（num_workers）
- 自动填充（collate_fn）

In [ ]:
# DataLoader使用演示
from torch.utils.data import DataLoader, TensorDataset

# 用TensorDataset做简单演示（不需要真实音频文件）
dummy_data = torch.randn(100, 16000)  # 100个样本，每个16000个采样点
dummy_labels = torch.randint(0, 10, (100,))  # 100个标签，10个类别

dataset = TensorDataset(dummy_data, dummy_labels)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

for batch_x, batch_y in loader:
    print(f'批量数据形状: {batch_x.shape}')  # [16, 16000]
    print(f'批量标签形状: {batch_y.shape}')  # [16]
    break

## 本节要点

1. **类 = 数据 + 方法**，属性（self.xxx）存储数据，方法操作数据
2. **继承**就是在父类的基础上加修改，用 `super().__init__()` 调用父类构造函数
3. **nn.Module** 是PyTorch的基石——只需实现 `__init__`（定义层）和 `forward`（定义数据流）
4. **Dataset** 只需实现 `__len__` 和 `__getitem__`，DataLoader自动处理批量、打乱等细节


---
下一节：[03-debugging-git.ipynb](03-debugging-git.ipynb) — 调试、版本控制与编程习惯